In [2]:
!pip install geopandas folium pyogrio shapely

In [3]:
# import geopandas as gpd

# file_path = r"C:\Users\ShriyansJain\Downloads\tl_2020_us_zcta520\tl_2020_us_zcta520.shp"   # update path if needed

# gdf = gpd.read_file(file_path)

# print(gdf.shape)
# gdf.head()

In [4]:
# gdf = gdf[["ZCTA5CE20", "geometry"]]
# gdf = gdf.rename(columns={"ZCTA5CE20": "ZIP"})

# gdf.head()

In [5]:
# gdf["geometry"] = gdf.geometry.simplify(0.001)

In [6]:
# gdf.plot(figsize=(16, 10))

In [7]:
# import folium

# m = folium.Map(location=[39.5, -98.35], zoom_start=4)

# folium.GeoJson(
#     gdf,
#     tooltip=folium.GeoJsonTooltip(fields=["ZIP"])
# ).add_to(m)

# m

In [8]:
import geopandas as gpd
import pandas as pd
import folium


file_path = r"C:\Users\ShriyansJain\Downloads\tl_2020_us_zcta520\tl_2020_us_zcta520.shp"   # update path if needed

gdf = gpd.read_file(file_path)

print("Original shape:", gdf.shape)

# Keep only required columns
gdf = gdf[["ZCTA5CE20", "geometry"]].copy()
gdf = gdf.rename(columns={"ZCTA5CE20": "ZIP"})

# Simplify geometry for faster rendering
gdf["geometry"] = gdf.geometry.simplify(0.001)

print("Processed shape:", gdf.shape)

gdf.head()

Original shape: (33791, 10)
Processed shape: (33791, 2)


,ZIP,geometry
0,35592,"POLYGON ((-88.24735 33.6539, -88.24489 33.6570..."
1,35616,"POLYGON ((-88.13997 34.58184, -88.11971 34.714..."
2,35621,"POLYGON ((-86.81659 34.3496, -86.81207 34.3534..."
3,35651,"POLYGON ((-87.53096 34.41468, -87.53042 34.470..."
4,36010,"POLYGON ((-85.95712 31.67744, -85.94678 31.676..."


In [9]:
# -----------------------------
# CREATE ZIP MASTER TABLE
# -----------------------------
centroids = gdf.geometry.centroid

zip_master = pd.DataFrame({
    "ZIP": gdf["ZIP"],
    "LAT": centroids.y,
    "LON": centroids.x,
    "TERRITORY_ID": "UNASSIGNED",
    "SALES": 0,
    "CLAIMS": 0,
    "HCP_COUNT": 0
})

zip_master.head()

C:\Users\ShriyansJain\AppData\Local\Temp\ipykernel_36104\1350553190.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


,ZIP,LAT,LON,TERRITORY_ID,SALES,CLAIMS,HCP_COUNT
0,35592,33.743098,-88.096702,UNASSIGNED,0,0,0
1,35616,34.738264,-88.018852,UNASSIGNED,0,0,0
2,35621,34.349664,-86.723913,UNASSIGNED,0,0,0
3,35651,34.462469,-87.480430,UNASSIGNED,0,0,0
4,36010,31.667481,-85.814413,UNASSIGNED,0,0,0


In [10]:
# -----------------------------
# TOOL FUNCTIONS
# -----------------------------
def assign_zip(zip_code, territory_id):
    zip_master.loc[
        zip_master["ZIP"] == str(zip_code),
        "TERRITORY_ID"
    ] = territory_id
    print(f"ZIP {zip_code} assigned to {territory_id}")


def get_zip_info(zip_code):
    return zip_master[zip_master["ZIP"] == str(zip_code)]


def evaluate_balance():
    return (
        zip_master
        .groupby("TERRITORY_ID")
        .agg({
            "CLAIMS": "sum",
            "SALES": "sum",
            "ZIP": "count"
        })
        .rename(columns={"ZIP": "ZIP_COUNT"})
        .reset_index()
    )

In [11]:
assign_zip("83278", "T1")
assign_zip("83279", "T1")
assign_zip("83280", "T2")

evaluate_balance()

ZIP 83278 assigned to T1
ZIP 83279 assigned to T1
ZIP 83280 assigned to T2


,TERRITORY_ID,CLAIMS,SALES,ZIP_COUNT
0,T1,0,0,1
1,UNASSIGNED,0,0,33790


In [12]:
# -----------------------------
# MERGE MAP + STATE
# -----------------------------
map_df = gdf.merge(
    zip_master[["ZIP", "TERRITORY_ID"]],
    on="ZIP",
    how="left"
)

map_df.head()

,ZIP,geometry,TERRITORY_ID
0,35592,"POLYGON ((-88.24735 33.6539, -88.24489 33.6570...",UNASSIGNED
1,35616,"POLYGON ((-88.13997 34.58184, -88.11971 34.714...",UNASSIGNED
2,35621,"POLYGON ((-86.81659 34.3496, -86.81207 34.3534...",UNASSIGNED
3,35651,"POLYGON ((-87.53096 34.41468, -87.53042 34.470...",UNASSIGNED
4,36010,"POLYGON ((-85.95712 31.67744, -85.94678 31.676...",UNASSIGNED


In [14]:
# # -----------------------------
# # TERRITORY COLOR MAP
# # -----------------------------
# color_map = {
#     "UNASSIGNED": "gray",
#     "T1": "blue",
#     "T2": "green",
#     "T3": "red"
# }

# m = folium.Map(location=[39.5, -98.35], zoom_start=4)

# for _, row in map_df.iterrows():
#     folium.GeoJson(
#         row["geometry"],
#         style_function=lambda feature, territory=row["TERRITORY_ID"]: {
#             "fillColor": color_map.get(territory, "gray"),
#             "color": "black",
#             "weight": 0.5,
#             "fillOpacity": 0.5
#         },
#         tooltip=f"ZIP: {row['ZIP']} | Territory: {row['TERRITORY_ID']}"
#     ).add_to(m)

# m